# 01 — Dataset EDA, Cleaning and Bias Screening
Run this notebook after extracting the assignment dataset into `data/raw/<class_name>/`. It produces data counts and quality-analysis plots. Outlier flags are review candidates, not automatic labels.

In [ ]:
from pathlib import Path
import sys, os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/"src").exists() and (ROOT.parent/"src").exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
DATA_DIR = ROOT/"data"/"raw"
OUTPUT_DIR = ROOT/"outputs"
for d in ["plots","confusion_matrices","metrics","predictions"]: (OUTPUT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Project root:", ROOT, "| Dataset:", DATA_DIR)

## Optional dataset download
If Google Drive permits direct downloads, run the following cell. If it fails, download manually from the assignment link and extract the archive into `data/raw/`.

In [ ]:
# Optional: download file from assignment Google Drive link.
# !pip -q install gdown
# !python download_dataset.py
# Inspect the downloaded file and extract it manually if it is an archive.

In [ ]:
from src.dataset import discover_dataset
df = discover_dataset(DATA_DIR, validate_images=False, require_all_classes=False)
print("Unique image candidates:", len(df))
counts = df["label"].value_counts().reindex(["underripe","breaking","ripe","overripe"], fill_value=0)
display(counts.rename("image_count").to_frame())
if counts.sum():
    print("Largest/smallest class ratio:", round(counts.max() / max(counts[counts > 0].min(), 1), 2))
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind="bar", ax=ax)
ax.set(title="Avocado images by class", xlabel="Ripening stage", ylabel="Image count")
plt.xticks(rotation=25); plt.tight_layout()
fig.savefig(OUTPUT_DIR/"plots"/"class_distribution.png", dpi=160); plt.show()

In [ ]:
import cv2
from PIL import Image
rows=[]; bad=[]
label_by_path = dict(zip(df.path, df.label))
for p in df.path:
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            rgb=np.asarray(im.convert("RGB"))
        h,w=rgb.shape[:2]
        hsv=cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
        gray=cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        border=max(1,min(h,w)//20)
        border_pixels=np.concatenate([rgb[:border].reshape(-1,3),rgb[-border:].reshape(-1,3),
                                      rgb[:, :border].reshape(-1,3),rgb[:, -border:].reshape(-1,3)])
        rows.append({"path":p,"label":label_by_path[p],
                     "width":w,"height":h,"aspect_ratio":w/h,
                     "brightness_mean":float(gray.mean()),
                     "brightness_std":float(gray.std()),
                     "dark_pixel_fraction":float((gray<20).mean()),
                     "clipped_bright_fraction":float((gray>245).mean()),
                     "saturation_mean":float(hsv[:,:,1].mean()),
                     "red_mean":float(rgb[:,:,0].mean()),"green_mean":float(rgb[:,:,1].mean()),
                     "blue_mean":float(rgb[:,:,2].mean()),
                     "laplacian_variance":float(cv2.Laplacian(gray,cv2.CV_64F).var()),
                     "border_red_mean":float(border_pixels[:,0].mean()),
                     "border_green_mean":float(border_pixels[:,1].mean()),
                     "border_blue_mean":float(border_pixels[:,2].mean()),
                     "border_rgb_std":float(border_pixels.std()),
                     "file_bytes":Path(p).stat().st_size})
    except (OSError, ValueError, cv2.error) as e: bad.append({"path":p,"error":str(e)})
quality=pd.DataFrame(rows)
quality.to_csv(ROOT/"data"/"processed"/"image_quality.csv",index=False)
pd.DataFrame(bad).to_csv(ROOT/"data"/"processed"/"unreadable_images.csv",index=False)
print("Valid:",len(quality),"Unreadable:",len(bad))
if quality.empty: raise ValueError("No readable images remain. See unreadable_images.csv.")
display(quality.groupby("label")[["width","height","brightness_mean","saturation_mean",
    "dark_pixel_fraction","clipped_bright_fraction","laplacian_variance","border_rgb_std"]].agg(["mean","std","min","max"]).round(2))

In [ ]:
fig, axes=plt.subplots(1,2,figsize=(12,4))
quality.boxplot(column="width",by="label",ax=axes[0]); axes[0].set_title("Image width by class")
quality.boxplot(column="brightness_mean",by="label",ax=axes[1]); axes[1].set_title("Mean brightness by class")
fig.suptitle(""); plt.tight_layout(); fig.savefig(OUTPUT_DIR/"plots"/"resolution_brightness.png",dpi=160); plt.show()

In [ ]:
# Sample grid
sample=df[df.path.isin(quality.path)].groupby("label",group_keys=False).head(4)
fig,axes=plt.subplots(max(1,len(sample)),1,figsize=(10,3*max(1,len(sample))))
if len(sample)==1: axes=[axes]
for ax,(_,row) in zip(axes,sample.iterrows()):
    img=Image.open(row.path).convert("RGB")
    ax.imshow(img); ax.set_title(f"{row.label}: {Path(row.path).name}"); ax.axis("off")
plt.tight_layout(); fig.savefig(OUTPUT_DIR/"plots"/"sample_images.png",dpi=130); plt.show()

In [ ]:
# Heuristic outlier flags using within-class robust z-scores for brightness and saturation.
def robust_z(s):
    med=s.median(); mad=(s-med).abs().median()
    return (s-med)/(1.4826*mad+1e-8)
outlier_columns=["width","height","brightness_mean","saturation_mean","laplacian_variance","border_rgb_std"]
for column in outlier_columns:
    quality[f"{column}_robust_z"]=quality.groupby("label")[column].transform(robust_z)
quality["review_outlier"]=quality[[f"{column}_robust_z" for column in outlier_columns]].abs().gt(3.5).any(axis=1)
quality.to_csv(ROOT/"data"/"processed"/"image_quality.csv",index=False)
display(quality[quality.review_outlier][["path","label","brightness_mean","saturation_mean","laplacian_variance","border_rgb_std","review_outlier"]].head(30))
print("Review candidates:",int(quality.review_outlier.sum()))

## Interpretation checklist
- Class imbalance: compare class counts and report the largest/smallest ratio.
- Lighting: inspect low/high brightness and saturation candidates visually.
- Resolution: inspect width, height and aspect ratio; resize at model input.
- Background/camera bias: check whether class labels correlate with background or capture session.
- Noise/labels: inspect duplicates, blur, occlusion, partial fruit, and questionable labels manually.
Do not remove outliers without a documented reason. Exported plots and `image_quality.csv` are evidence for the report.